# **Notebook 10 – Feature Importance**

In [1]:
# Load Datset
import pandas as pd
df = pd.read_csv("hospital_patient_readmission.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Patient_ID            2500 non-null   object 
 1   Age                   2500 non-null   int64  
 2   Gender                2500 non-null   object 
 3   BMI                   2492 non-null   float64
 4   Blood_Pressure        2494 non-null   float64
 5   Department            2500 non-null   object 
 6   Admission_Type        2500 non-null   object 
 7   Diagnosis             2495 non-null   object 
 8   Length_of_Stay        2500 non-null   int64  
 9   Previous_Visits       2500 non-null   int64  
 10  Medication_Count      2500 non-null   int64  
 11  Lab_Test_Count        2500 non-null   int64  
 12  Treatment_Type        2500 non-null   object 
 13  Insurance_Type        2500 non-null   object 
 14  Doctor_Experience     2500 non-null   int64  
 15  Patient_Satisfaction 

## **1. What is Feature Importance?**

**Understand the Concept**

* Feature Importance shows which features are more useful for the model.



## **2. Model-Based Feature Importance**

**Understand the Concept**

* Model-based feature importance uses a trained model to find important features.
* It shows which features help the model make predictions.


## **3. Decision Tree Feature Importance**

**Understand the Concept**

* A Decision Tree can show which features are important for making predictions.
* Features used more effectively in the tree get higher importance.

**Demonstrate the Concept**

* The model checks features such as Age, Previous_Visits, and Length_of_Stay.
* It gives each feature an importance score.

In [3]:
from sklearn.preprocessing import LabelEncoder

data = df.copy()

for column in data.select_dtypes(include="object").columns:
    data[column] = LabelEncoder().fit_transform(data[column].astype(str))

X = data.drop(columns=["Patient_ID", "Readmitted"])
y = data["Readmitted"]

In [4]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X, y)

tree_importance = pd.Series(tree_model.feature_importances_,index=X.columns)
print(tree_importance.sort_values(ascending=False))

Treatment_Cost          0.125790
Age                     0.116897
BMI                     0.095701
Follow_Up_Days          0.092042
Blood_Pressure          0.088782
Doctor_Experience       0.063736
Length_of_Stay          0.058548
Lab_Test_Count          0.056929
Previous_Visits         0.056028
Patient_Satisfaction    0.046766
Medication_Count        0.039605
Diagnosis               0.037398
Treatment_Type          0.034634
Department              0.031244
Insurance_Type          0.028458
Admission_Type          0.017633
Gender                  0.009810
dtype: float64


**Explanation**

* The Decision Tree gives an importance score to each feature.
* Higher scores show features that are more useful to the tree.

**Justify Your Feature**

* I used Decision Tree importance to find the features that help predict Readmitted.

## **4. Random Forest Feature Importance**

**Understand the Concept**

* Random Forest uses many Decision Trees.
* It combines their results to find important features.

**Demonstrate the Concept**

* Features with higher importance scores are more useful to the Random Forest model.

In [5]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X, y)

rf_importance = pd.Series(rf_model.feature_importances_, index=X.columns)
print(rf_importance.sort_values(ascending=False))

Treatment_Cost          0.103302
BMI                     0.096093
Blood_Pressure          0.090674
Age                     0.087864
Follow_Up_Days          0.086532
Doctor_Experience       0.072521
Lab_Test_Count          0.067630
Length_of_Stay          0.066967
Patient_Satisfaction    0.057652
Medication_Count        0.054786
Previous_Visits         0.049264
Diagnosis               0.038954
Department              0.036116
Treatment_Type          0.030196
Insurance_Type          0.023889
Admission_Type          0.022266
Gender                  0.015296
dtype: float64


**Explanation**

* Random Forest gives an importance score to each feature.
* Higher scores show more important features.

**Justify Your Feature**

* I used Random Forest to find the features that are most useful for predicting Readmitted.

## **5. Permutation Importance**

**Understand the Concept**

* Permutation Importance checks how important a feature is by shuffling its values.
* If the model performs worse after shuffling, that feature is important.

**Demonstrate the Concept**

* Shuffle one feature at a time.
* Check how much the model performance changes.

In [12]:
from sklearn.inspection import permutation_importance

result = permutation_importance(rf_model, X, y, random_state=42, n_repeats=5)
permutation_importance_df = pd.DataFrame({ "Feature": X.columns,"Importance": result.importances_mean}).sort_values("Importance", ascending=False)
print(permutation_importance_df)

                 Feature  Importance
7         Length_of_Stay     0.08904
10        Lab_Test_Count     0.07584
16        Treatment_Cost     0.05800
0                    Age     0.04064
14  Patient_Satisfaction     0.03912
15        Follow_Up_Days     0.03824
3         Blood_Pressure     0.03768
9       Medication_Count     0.03736
2                    BMI     0.03656
13     Doctor_Experience     0.03008
8        Previous_Visits     0.02160
6              Diagnosis     0.01216
5         Admission_Type     0.00472
4             Department     0.00408
11        Treatment_Type     0.00384
12        Insurance_Type     0.00168
1                 Gender     0.00016


**Explanation**

* Each feature is shuffled one at a time.
* A bigger performance drop means higher importance.

**Justify Your Feature**

* I used Permutation Importance to check how much each feature affects model performance.

## **6. Feature Importance vs Feature Selection**

**Understand the Concept**

* Feature Importance tells us how useful each feature is.
* Feature Selection chooses the features we want to keep.

**Demonstrate the Concept**

* Feature Importance gives scores to all features.
* Feature Selection keeps only the most useful features.

In [8]:
# Feature Importance
importance = pd.Series(rf_model.feature_importances_,index=X.columns).sort_values(ascending=False)
print(rf_importance.sort_values(ascending=False))

Treatment_Cost          0.103302
BMI                     0.096093
Blood_Pressure          0.090674
Age                     0.087864
Follow_Up_Days          0.086532
Doctor_Experience       0.072521
Lab_Test_Count          0.067630
Length_of_Stay          0.066967
Patient_Satisfaction    0.057652
Medication_Count        0.054786
Previous_Visits         0.049264
Diagnosis               0.038954
Department              0.036116
Treatment_Type          0.030196
Insurance_Type          0.023889
Admission_Type          0.022266
Gender                  0.015296
dtype: float64


In [10]:
# Feature Selection - top 5
top_features = importance.head(5)
print(top_features)

Treatment_Cost    0.103302
BMI               0.096093
Blood_Pressure    0.090674
Age               0.087864
Follow_Up_Days    0.086532
dtype: float64


**Explanation**

* Feature importance gives a score to each feature.
* The top features can then be selected.



## **7. Interpreting Feature Importance**

**Understand the Concept**

* Feature importance shows how much a feature helps the model.
* A higher score means the feature is more useful to the model.

**Demonstrate the Concept**

* Compare the importance scores of all features.
* Features with higher scores are more important.

In [11]:
importance = pd.Series(rf_model.feature_importances_,index=X.columns).sort_values(ascending=False)
print(importance)

Treatment_Cost          0.103302
BMI                     0.096093
Blood_Pressure          0.090674
Age                     0.087864
Follow_Up_Days          0.086532
Doctor_Experience       0.072521
Lab_Test_Count          0.067630
Length_of_Stay          0.066967
Patient_Satisfaction    0.057652
Medication_Count        0.054786
Previous_Visits         0.049264
Diagnosis               0.038954
Department              0.036116
Treatment_Type          0.030196
Insurance_Type          0.023889
Admission_Type          0.022266
Gender                  0.015296
dtype: float64


**Explanation**

* The scores help us compare the features.
* Higher scores indicate greater importance.

**Justify Your Feature**

* I compared the scores to understand which features are most useful to the model.

## **8. Limitations of Feature Importance**

**Understand the Concept**

* Feature importance shows how useful a feature is to the model.
* High importance does not mean the feature causes the target.
* Importance can change with different models.
* Correlated features can share importance.
* Scores can sometimes be misleading.
* Importance does not explain why the feature is important.